# Smart Book: apni offline awaaz train karein

Ye notebook aap ke ElevenLabs wale jumlon (`smart-book-awaaz-data.zip`) se ek **offline awaaz** banata hai jo phone ke andar bina internet chalegi. Training Piper ke Hindi model (`hi_IN rohan medium`) se shuru hoti hai, is liye kam data mein bhi seekh leti hai.

**Pehle:**
1. Google Drive mein `SmartBook` naam ka folder banayein aur `smart-book-awaaz-data.zip` us mein daal dein.
2. Upar menu: **Runtime → Change runtime type → T4 GPU** → Save.
3. Har cell ko upar se neeche ek ek kar ke chalayein (▶ button).

**Colab band ho jaye** (muft Colab kuch ghante baad band kar deta hai): koi baat nahi. Training har kuch minute baad Drive mein mehfooz hoti rehti hai. Dobara notebook kholein, T4 GPU chunein, aur saare cells phir se chalayein. Training wahin se aage chalegi.

**Aakhir mein:** Drive ke `SmartBook` folder mein `smart-book-medium.onnx` aur `smart-book-medium.onnx.json` banenge. Ye dono files bhejni hain.

In [ ]:
#@title 1. Drive jorein aur settings
from google.colab import drive
drive.mount('/content/drive')

import os
WORK = '/content/drive/MyDrive/SmartBook'      # Drive ka folder
ZIP = WORK + '/smart-book-awaaz-data.zip'      # ElevenLabs wale jumle
EPOCHS = 1000     # kitni dafa poora data parhe. Zyada = behtar awaaz, lekin zyada waqt
BATCH = 16        # T4 GPU ke liye theek

assert os.path.exists(ZIP), 'Zip nahi mili: ' + ZIP + '  (Drive mein SmartBook folder aur zip check karein)'
os.makedirs(WORK + '/train', exist_ok=True)
print('Theek hai. Zip mil gayi.')
!nvidia-smi --query-gpu=name --format=csv,noheader || echo 'GPU nahi mila! Runtime -> Change runtime type -> T4 GPU'

In [ ]:
#@title 2. Piper install karein (5-10 minute)
!apt-get -qq update > /dev/null && apt-get -qq install -y espeak-ng build-essential cmake ninja-build > /dev/null
%cd /content
!test -d piper1-gpl || git clone -q https://github.com/OHF-voice/piper1-gpl.git
%cd /content/piper1-gpl
!pip -q install cython
!pip -q install -e '.[train]'
!./build_monotonic_align.sh
!python setup.py build_ext --inplace > /dev/null
!python -c "import piper.train; print('Piper tayyar')"

In [ ]:
#@title 3. Data kholein
import zipfile, csv
DATA = '/content/data'
if not os.path.exists(DATA + '/metadata_piper.csv'):
    zipfile.ZipFile(ZIP).extractall(DATA)
    rows = [l.rstrip('\n').split('|', 1) for l in open(DATA + '/metadata.csv', encoding='utf-8') if '|' in l]
    rows = [(i, t) for i, t in rows if os.path.exists(f'{DATA}/wavs/{i}.wav')]
    with open(DATA + '/metadata_piper.csv', 'w', encoding='utf-8') as f:
        for i, t in rows:
            f.write(f'{i}.wav|{t}\n')
n = sum(1 for _ in open(DATA + '/metadata_piper.csv', encoding='utf-8'))
print(n, 'jumle tayyar')
print(open(DATA + '/metadata_piper.csv', encoding='utf-8').readline())

In [ ]:
#@title 4. Hindi model (shuru ki jagah) download karein
BASE = '/content/hi_rohan.ckpt'
if not os.path.exists(BASE):
    !wget -q -O {BASE} 'https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/hi/hi_IN/rohan/medium/epoch%3D3190-step%3D309852.ckpt'
print(round(os.path.getsize(BASE) / 1e6), 'MB')

In [ ]:
#@title 5. Training (ghanton chalegi; Colab band ho to dobara saare cells chalayein)
import glob, shutil, subprocess, threading, time
SAVED = WORK + '/last.ckpt'          # Drive mein mehfooz training
LOCAL = '/content/train'

cmd = ['python', '-m', 'piper.train', 'fit',
       '--data.voice_name', 'smart-book',
       '--data.csv_path', DATA + '/metadata_piper.csv',
       '--data.audio_dir', DATA + '/wavs',
       '--model.sample_rate', '22050',
       '--data.espeak_voice', 'hi',
       '--data.cache_dir', '/content/cache',
       '--data.config_path', WORK + '/smart-book-medium.onnx.json',
       '--data.batch_size', str(BATCH),
       '--data.num_workers', '2',
       '--trainer.max_epochs', str(EPOCHS),
       '--trainer.default_root_dir', LOCAL]
if os.path.exists(SAVED):
    print('Pichli training mili, wahin se aage...')
    cmd += ['--ckpt_path', SAVED]
else:
    print('Nayi training, Hindi model se shuru...')
    cmd += ['--model.warmstart_ckpt', BASE]

# har 10 minute baad aakhri training Drive mein copy (sirf ek file, taa ke Drive na bhare)
def keep_safe():
    while True:
        time.sleep(600)
        found = sorted(glob.glob(LOCAL + '/lightning_logs/version_*/checkpoints/last.ckpt'), key=os.path.getmtime)
        if found:
            shutil.copy(found[-1], SAVED + '.tmp'); os.replace(SAVED + '.tmp', SAVED)
            print(time.strftime('%H:%M'), 'Drive mein mehfooz kar diya', flush=True)
threading.Thread(target=keep_safe, daemon=True).start()

p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    if 'Epoch' in line or 'Error' in line or 'error' in line or 'Traceback' in line:
        print(line.rstrip()[-200:], flush=True)
p.wait()
found = sorted(glob.glob(LOCAL + '/lightning_logs/version_*/checkpoints/last.ckpt'), key=os.path.getmtime)
if found:
    shutil.copy(found[-1], SAVED)
print('Training khatam' if p.returncode == 0 else 'Training ruk gayi (code %s)' % p.returncode)

In [ ]:
#@title 6. Model banayein (training ke baad, ya beech mein sun-ne ke liye)
!python -m piper.train.export_onnx --checkpoint {SAVED} --output-file {WORK}/smart-book-medium.onnx
print('Ban gaya:', WORK + '/smart-book-medium.onnx')

In [ ]:
#@title 7. Sunein
from IPython.display import Audio, display
TEXT = 'पेट्रोल के पांच सौ रुपये लिख लिए। और कुछ?'  #@param {type:"string"}
!python -m piper -m {WORK}/smart-book-medium.onnx -c {WORK}/smart-book-medium.onnx.json -f /content/test.wav -- '{TEXT}'
display(Audio('/content/test.wav'))